##### Copyright 2026 Google LLC.

In [2]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multimodal Live API - Translation Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_LiveTranslate.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

**Preview**: The Live API is in preview.

This notebook demonstrates usage of the Gemini Live API for real-time audio translation. For an overview of new capabilities refer to the [Gemini Live API docs](https://ai.google.dev/gemini-api/docs/live-api/capabilities).

Some features of the API (such as low-latency bidirectional voice and video streaming using the local microphone and camera) are not supported in a standard Colab environment due to its headless cloud VM nature. To try full local hardware streaming, check out the CLI examples in the [Cookbook repository](https://github.com/google-gemini/cookbook/tree/main/quickstarts).

In this notebook, you will learn how to **stream and translate audio from a URL** in real-time using the Live Translation API, displaying live transcripts and playing the translated audio output.

## Setup

### Install SDK and Dependencies

The new **[Google Gen AI SDK](https://ai.google.dev/gemini-api/docs/sdks)** provides programmatic access to Gemini.

> **Note**: This notebook also uses `ffmpeg` to process the audio stream. `ffmpeg` is pre-installed in Google Colab environments.

In [3]:
%pip install -U -q google-genai

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for details.

In [4]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

The client will pick up your API key from the environment variable.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

### Select a model

The Live Translation API uses the translation-capable Live models.

In [6]:
MODEL = 'gemini-3.5-live-translate-preview'  # @param ['gemini-3.5-live-translate-preview'] {allow-input: true, isTemplate: true}

### Import Modules

Import the necessary packages for handling async events, audio format streams, and file writing.

In [7]:
import array
import asyncio
import contextlib
import wave

from IPython.display import display, Audio

from google import genai
from google.genai import types

### Helper to Write WAV Files

Let's define a helper context manager to write received audio chunks to a `.wav` file for playback in the notebook:

In [8]:
@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

## Audio URL Streaming & Translation

You can stream audio in real-time to the Live API, and receive translated audio back. Here, we'll stream audio from a public audio URL in 100ms chunks to mimic real-time audio input, and stream translation responses back.

### Helper for Streaming Audio URL

We define a helper function to stream audio from an HTTP URL and use `ffmpeg` to transcode it to raw PCM 16kHz mono audio.

In [9]:
async def stream_audio_url(url: str, audio_queue: asyncio.Queue, sample_rate: int = 16000, channels: int = 1, chunk_size: int = 1600):
    """Streams audio from an HTTP URL, decoding it via ffmpeg and putting raw PCM bytes into the audio_queue."""
    print(f"\n[Info] Starting audio stream via ffmpeg from: {url}")
    # Spawn ffmpeg to decode stream to raw PCM 16kHz mono 16-bit
    process = await asyncio.create_subprocess_exec(
        'ffmpeg',
        '-i', url,
        '-f', 's16le',
        '-acodec', 'pcm_s16le',
        '-ar', str(sample_rate),
        '-ac', str(channels),
        '-',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.DEVNULL
    )

    # 1600 samples * 2 bytes/sample (16-bit) = 3200 bytes per chunk
    chunk_size_bytes = chunk_size * 2
    bytes_per_second = sample_rate * 2
    start_time = asyncio.get_event_loop().time()
    bytes_sent = 0

    try:
        while True:
            data = await process.stdout.read(chunk_size_bytes)
            if not data:
                break

            await audio_queue.put(data)
            bytes_sent += len(data)

            # Rate limit to real-time speed (1.0x) so we simulate real mic streaming
            expected_elapsed = bytes_sent / bytes_per_second
            actual_elapsed = asyncio.get_event_loop().time() - start_time
            sleep_time = expected_elapsed - actual_elapsed
            if sleep_time > 0:
                await asyncio.sleep(sleep_time)
    except asyncio.CancelledError:
        pass
    finally:
        if process.returncode is None:
            try:
                process.terminate()
                await process.wait()
            except Exception:
                pass
        print("\n[Info] Audio stream finished.")

### Helper for Sending Audio & Receiving Translated Responses

Next, we define functions to push the audio chunks from the queue to the Live session, and to receive and print the source and translation transcripts.

In [10]:
async def send_realtime(session, audio_queue: asyncio.Queue, sample_rate: int = 16000):
    """Sends audio from the input queue to the GenAI session."""
    try:
        while True:
            chunk = await audio_queue.get()
            await session.send_realtime_input(
                audio=types.Blob(
                    data=chunk,
                    mime_type=f"audio/pcm;rate={sample_rate}"
                )
            )
            audio_queue.task_done()
    except asyncio.CancelledError:
        pass

### Run Audio URL Translation

Now, we set up our main translation runner. We'll use a static audio URL: `https://storage.googleapis.com/generativeai-downloads/gemini-cookbook/audio/gemini-live-translate-sample.wav`.
The code runs the input audio stream, upload stream, and receiver concurrently in a `TaskGroup`. The translated Spanish audio is written to a wave file and played back.

In [11]:
async def run_audio_translation(url: str, target_lang: str):
    # Configure live connection with translation settings
    config = types.LiveConnectConfig(
        response_modalities=[types.Modality.AUDIO],
        translation_config=types.TranslationConfig(
            echo_target_language=True,
            target_language_code=target_lang,
        ),
        input_audio_transcription=types.AudioTranscriptionConfig(),
        output_audio_transcription=types.AudioTranscriptionConfig(),
    )

    audio_queue_input = asyncio.Queue(maxsize=10)
    file_name = 'audio_translation.wav'

    print(f"[Info] Connecting to Gemini Live ({MODEL})...")

    try:
        async with client.aio.live.connect(model=MODEL, config=config) as session:
            print("[Info] Connected successfully. Starting stream...")

            with wave_file(file_name) as wav:
                async def receive_responses():
                    try:
                        async for response in session.receive():
                            server_content = response.server_content
                            if server_content:
                                # Write translated audio chunks to WAV file
                                if server_content.model_turn:
                                    for part in server_content.model_turn.parts:
                                        if part.inline_data and isinstance(part.inline_data.data, bytes):
                                            wav.writeframes(part.inline_data.data)

                                # Print input (source) transcript
                                if server_content.input_transcription and server_content.input_transcription.text:
                                    lang = f" ({server_content.input_transcription.language_code})" if server_content.input_transcription.language_code else ""
                                    print(f"\n[Source{lang}] {server_content.input_transcription.text}", flush=True)

                                # Print output (translated) transcript
                                if server_content.output_transcription and server_content.output_transcription.text:
                                    lang = f" ({server_content.output_transcription.language_code})" if server_content.output_transcription.language_code else ""
                                    print(f"[Translation{lang}] {server_content.output_transcription.text}", flush=True)
                    except asyncio.CancelledError:
                        pass
                    except Exception as e:
                        print(f"\n[Error] Receiving loop encountered error: {e}")

                async with asyncio.TaskGroup() as tg:
                    # Task 1: Stream original audio from URL to a queue
                    stream_task = tg.create_task(
                        stream_audio_url(url, audio_queue_input)
                    )
                    # Task 2: Upload audio queue to Gemini Live Translate API
                    send_task = tg.create_task(send_realtime(session, audio_queue_input))
                    # Task 3: Receive transcripts and translated audio response chunks
                    receive_task = tg.create_task(receive_responses())

                    # Wait for the audio stream to finish reading
                    await stream_task

                    # Wait for all buffered input chunks to be sent to Gemini
                    await audio_queue_input.join()
                    send_task.cancel()

                    # Give Gemini a few seconds to finish translating the final chunks
                    await asyncio.sleep(4.0)
                    receive_task.cancel()

    except Exception as e:
        print(f"[Error] Live session error: {e}")

    print("\nTranslation complete!")
    display(Audio(filename=file_name, autoplay=True))

#@title Run Translation { run: "auto" }
# Audio: Gemini Live Translate sample (or enter your own public audio URL)
audio_url = "https://storage.googleapis.com/generativeai-downloads/gemini-cookbook/audio/gemini-live-translate-sample.wav" #@param {type:"string"}
target_lang = "jp" #@param {type:"string"}

await run_audio_translation(
    url=audio_url,
    target_lang=target_lang
)

[Info] Connecting to Gemini Live (gemini-3.5-live-translate-preview)...
[Info] Connected successfully. Starting stream...

[Info] Starting audio stream via ffmpeg from: https://storage.googleapis.com/generativeai-downloads/gemini-cookbook/audio/gemini-live-translate-sample.wav

[Error] Receiving loop encountered error: 1007 None. Request contains an invalid argument.

[Info] Audio stream finished.
[Error] Live session error: unhandled errors in a TaskGroup (1 sub-exception)

Translation complete!


## Next steps

This tutorial shows basic audio translation capabilities using the Multimodal Live API.

- Try it out in [Google AI Studio](https://aistudio.google.com/live?model=gemini-3.5-live-translate-preview)
- Read the [docs](https://ai.google.dev/gemini-api/docs/live-api/live-translate)
- Clone the [Live API examples from GitHub](https://github.com/google-gemini/gemini-live-api-examples)
